# Apply Naive INT4 Weight-only Quantization on Qwen-7b-chat

In [1]:
import tqdm
import torch
from torch import nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from functools import partial
import gc
import os
os.environ['https_proxy'] = 'http://192.168.1.12:7891'

debug = True

if debug:
    # improve torch tensor printing
    import torch
    def custom_repr(self):
        return f'{{Tensor:{tuple(self.shape)}}} {original_repr(self)}'
    original_repr = torch.Tensor.__repr__
    torch.Tensor.__repr__ = custom_repr

/root/workspace/qwen_cpu_deployment/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Here we use wikitext-2 dataset for evaluation. The dataset is automatically downloaded by the code.

In [2]:
print("Loding wikitext datasets ...")
testenc = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test', cache_dir="~/.cache/huggingface/datasets")
print("Done")

Loding wikitext datasets ...
Done


In [3]:

def evaluate(model, testenc, tokenizer):
    # we control the text length to avoid error posed by tiktoken
    testenc = tokenizer("\n\n".join(testenc['text']), return_tensors='pt')
    testenc = testenc.input_ids.to(model.device)
    nsamples = 40
    model = model.eval()

    nlls = []
    for i in tqdm.tqdm(range(nsamples), desc="evaluating Qwen on wikitext"):
        batch = testenc[:, (i * 1024):((i + 1) * 1024)].to(model.device)
        with torch.no_grad():
            lm_logits = model(batch).logits
        shift_logits = lm_logits[:, :-1, :].contiguous().float()
        shift_labels = testenc[:, (i * 1024):((i + 1) * 1024)][:, 1:]
        loss_fct = nn.CrossEntropyLoss()
        loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        neg_log_likelihood = loss.float() * 1024
        nlls.append(neg_log_likelihood)

    return torch.exp(torch.stack(nlls).sum() / (nsamples * 1024))

def get_model_size(model: nn.Module, data_width=16, group_size=-1):
    # store the quantization parameters: 1 fp16 scaling factors and 1 int4 zero point
    # this might not be precise
    if group_size != -1:
        data_width += (16 + 4) / group_size
    num_elements = 0
    for param in model.parameters():
        num_elements += param.numel()
    return num_elements * data_width

Byte = 8
KiB = 1024 * Byte
MiB = 1024 * KiB
GiB = 1024 * MiB

# Evaluate the performance of FP32 Qwen
## PPL

In [4]:
model_path = "Qwen/Qwen-7B-Chat"
tokenizer = AutoTokenizer.from_pretrained(
    model_path, trust_remote_code=True
    )
model = AutoModelForCausalLM.from_pretrained(
    model_path, device_map="cuda", trust_remote_code=True, fp16=True
)

from transformers import AutoConfig, AutoTokenizer

config = AutoConfig.from_pretrained(model_path,trust_remote_code=True)

print(config.vocab_size)  # >> 32032
print(len(tokenizer))     # >> 32002

Try importing flash-attention for faster inference...
Loading checkpoint shards: 100%|██████████| 8/8 [00:03<00:00,  2.53it/s]


151936
151851


In [5]:
fp32_perplexity = evaluate(model, testenc, tokenizer)
print(f"\nmodel perplexity: {fp32_perplexity:.2f}")


Token indices sequence length is longer than the specified maximum sequence length for this model (299078 > 32768). Running this sequence through the model will result in indexing errors
evaluating Qwen on wikitext: 100%|██████████| 40/40 [00:04<00:00,  8.42it/s]


model perplexity: 10.07


In [6]:
model_size = get_model_size(model, data_width=32, group_size=-1)
print(f"model size: {model_size/MiB:.2f} MiB")

model size: 29454.52 MiB


In [7]:
model

QWenLMHeadModel(
  (transformer): QWenModel(
    (wte): Embedding(151936, 4096)
    (drop): Dropout(p=0.0, inplace=False)
    (rotary_emb): RotaryEmbedding()
    (h): ModuleList(
      (0-31): 32 x QWenBlock(
        (ln_1): RMSNorm()
        (attn): QWenAttention(
          (c_attn): Linear(in_features=4096, out_features=12288, bias=True)
          (c_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (attn_dropout): Dropout(p=0.0, inplace=False)
        )
        (ln_2): RMSNorm()
        (mlp): QWenMLP(
          (w1): Linear(in_features=4096, out_features=11008, bias=False)
          (w2): Linear(in_features=4096, out_features=11008, bias=False)
          (c_proj): Linear(in_features=11008, out_features=4096, bias=False)
        )
      )
    )
    (ln_f): RMSNorm()
  )
  (lm_head): Linear(in_features=4096, out_features=151936, bias=False)
)

In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()

## GSM8k

In [ ]:
# NOTE: Qwen-7b-chat tends to have bug for running this command
# !lm-eval --tasks gsm8k --model vllm --model_args pretrained=Qwen/Qwen-7B-Chat,max_model_len=8192 --batch_size auto --trust_remote_code

INFO 05-30 08:02:59 [__init__.py:243] Automatically detected platform cuda.
2025-05-30:08:03:02 INFO     [__main__:428] Passed `--trust_remote_code`, setting environment variable `HF_DATASETS_TRUST_REMOTE_CODE=true`
2025-05-30:08:03:02 INFO     [__main__:440] Selected Tasks: ['gsm8k']
2025-05-30:08:03:02 INFO     [evaluator:185] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-05-30:08:03:02 INFO     [evaluator:223] Initializing vllm model, with arguments: {'pretrained': 'Qwen/Qwen-7B-Chat', 'max_model_len': 8192, 'trust_remote_code': True}
INFO 05-30 08:03:02 [__init__.py:31] Available plugins for group vllm.general_plugins:
INFO 05-30 08:03:02 [__init__.py:33] - lora_filesystem_resolver -> vllm.plugins.lora_resolvers.filesystem_resolver:register_filesystem_resolver
INFO 05-30 08:03:02 [__init__.py:36] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.
INFO 0

## Evaluate Performance Of Int4 Weight-quantized Model
Apply pseudo quantization to check the performance of quantized model directly

In [ ]:
def pseudo_quantize_tensor_q4_zero_point(w, n_bit=4, q_group_size = -1):
    assert q_group_size == 32, "In this code, the quantize shape should be set to 32 to be compatible with tinychatengine code"

    org_w_shape = w.shape
    if q_group_size > 0:
        assert org_w_shape[-1] % q_group_size == 0
        w = w.reshape(-1, q_group_size)
        
    # get the max and min value of each group
    max_val = torch.amax(w, dim=-1, keepdim=True)
    min_val = torch.amin(w, dim=-1, keepdim=True)
    max_int = 2 ** n_bit - 1
    min_int = 0

    # get the scaling factor, zero point
    scaling_factor = (max_val - min_val).clamp(min=1e-5) / (max_int - min_int) # (len, 1)
    zero_point = (-torch.round(min_val / scaling_factor)).clamp_(0, max_int)
    # make sure no nan occurs
    assert (not scaling_factor.isnan().any())
    assert (not zero_point.isnan().any())
    
    # start the process of pseudo quantization
    # 1. quantize
    w_q =  torch.round( w / scaling_factor) + zero_point
    w_q = w_q.clamp(min_int, max_int)
    assert w_q.dim() == 2 and w_q.size(1) == q_group_size and w_q.size(0) == scaling_factor.size(0), \
            f"{w_q.size()} != {scaling_factor.size()}"
    # 2. dequantize
    w_f = (w_q - zero_point) * scaling_factor
    assert w_f.size() == w.size()
    assert (not w_f.isnan().any())
    max_error = (w_f - w).abs().max()
    w_f = w_f.reshape(org_w_shape)
    # print(f"Pseudo quantization error: {max_error}")
    return w_f 

def pseudo_quantize_tensor_q40(w, n_bit=4, q_group_size = -1):
    assert q_group_size == 32, "In this code, the quantize shape should be set to 32 to be compatible with tinychatengine code"

    org_w_shape = w.shape
    if q_group_size > 0:
        assert org_w_shape[-1] % q_group_size == 0
        w = w.reshape(-1, q_group_size)
        
    # get the max and min value of each group
    max_abs_value = torch.amax(w.abs(), dim=-1, keepdim=True)
    max_int = (2 ** (n_bit - 1) - 1)
    min_int = - (2 ** (n_bit - 1) )

    # get the scaling factor, zero point
    scaling_factor = (max_abs_value).clamp(min=1e-5) / (min_int) # (len, 1)
    zero_point = torch.tensor(8).round()
    # make sure no nan occurs
    assert (not scaling_factor.isnan().any())
    assert (not zero_point.isnan().any())
    
    # start the process of pseudo quantization
    # 1. quantize
    w_q =  torch.round( w / scaling_factor)
    w_q = w_q.clamp(min_int, max_int)
    assert w_q.dim() == 2 and w_q.size(1) == q_group_size and w_q.size(0) == scaling_factor.size(0), \
            f"{w_q.size()} != {scaling_factor.size()}"
    # 2. dequantize
    w_f = (w_q) * scaling_factor
    assert w_f.size() == w.size()
    assert (not w_f.isnan().any())
    max_error = (w_f - w).abs().max()
    w_f = w_f.reshape(org_w_shape)
    # print(f"Pseudo max quantization error: {max_error}")
    return w_f

def pseudo_quantize_tensor_q41(w, n_bit=4, q_group_size = -1):
    assert q_group_size == 32, "In this code, the quantize shape should be set to 32 to be compatible with tinychatengine code"

    org_w_shape = w.shape
    if q_group_size > 0:
        assert org_w_shape[-1] % q_group_size == 0
        w = w.reshape(-1, q_group_size)
        
    # get the max and min value of each group
    max_val = torch.amax(w, dim=-1, keepdim=True)
    min_val = torch.amin(w, dim=-1, keepdim=True)
    max_int = 2 ** n_bit - 1
    min_int = 0

    # get the scaling factor, zero point
    scaling_factor = (max_val - min_val).clamp(min=1e-5) / (max_int - min_int) # (len, 1)
    # make sure no nan occurs
    assert (not scaling_factor.isnan().any())
    
    # start the process of pseudo quantization
    # 1. quantize
    w_q = torch.round( 
            (w - min_val) / scaling_factor
        )
    w_q = w_q.clamp(min_int, max_int)
    assert w_q.dim() == 2 and w_q.size(1) == q_group_size and w_q.size(0) == scaling_factor.size(0), \
            f"{w_q.size()} != {scaling_factor.size()}"
    # 2. dequantize
    w_f = (w_q) * scaling_factor + min_val
    assert w_f.size() == w.size()
    assert (not w_f.isnan().any())
    max_error = (w_f - w).abs().max()
    w_f = w_f.reshape(org_w_shape)
    # print(f"Pseudo quantization error: {max_error}")
    return w_f 

quantization_method_dict = {
    "q4z": pseudo_quantize_tensor_q4_zero_point,
    "q40": pseudo_quantize_tensor_q40,
    "q41": pseudo_quantize_tensor_q41
}

@torch.no_grad()
def pseudo_quantize_model_weight(model, w_bit, q_group_size, method):
    q_method = quantization_method_dict[method]
    for n, m in model.named_modules():
        if isinstance(m, nn.Linear):
            # print(f"Pseudo quantizing {n}")
            m.weight.data = q_method(m.weight.data, w_bit, q_group_size)
            # print("")


In [ ]:
def quantize_and_evaluate(q_method):
    model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat", device_map="cuda", trust_remote_code=True, fp16=True).eval()
    # Apply fake quantization
    pseudo_quantize_model_weight(model, 4, 32, q_method)
    # Evaluate the model
    model_perplexity = evaluate(model, testenc, tokenizer)
    model_size = get_model_size(model, data_width=4, group_size=32)
    print(f"\nmodel perplexity: {model_perplexity:.2f}")
    print(f"model size: {model_size/MiB:.2f} MiB")
    return model


In [ ]:
gc.collect()
torch.cuda.empty_cache()

model = quantize_and_evaluate("q41")
quantized_model_path = "tmp_quantized_model"
model.save_pretrained(quantized_model_path)
tokenizer.save_pretrained(quantized_model_path)

del model
gc.collect()
torch.cuda.empty_cache()
!lm-eval --tasks gsm8k --model vllm --model_args pretrained=tmp_quantized_model,max_model_len=8192 --batch_size auto --trust_remote_code

Try importing flash-attention for faster inference...
evaluating Qwen on wikitext: 100%|██████████| 40/40 [00:04<00:00,  8.84it/s]



model perplexity: 10.61
model size: 4257.10 MiB
INFO 05-30 08:21:50 [__init__.py:243] Automatically detected platform cuda.
2025-05-30:08:21:53 INFO     [__main__:428] Passed `--trust_remote_code`, setting environment variable `HF_DATASETS_TRUST_REMOTE_CODE=true`
2025-05-30:08:21:53 INFO     [__main__:440] Selected Tasks: ['gsm8k']
2025-05-30:08:21:53 INFO     [evaluator:185] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-05-30:08:21:53 INFO     [evaluator:223] Initializing vllm model, with arguments: {'pretrained': 'tmp_quantized_model', 'max_model_len': 8192, 'trust_remote_code': True}
INFO 05-30 08:21:53 [__init__.py:31] Available plugins for group vllm.general_plugins:
INFO 05-30 08:21:53 [__init__.py:33] - lora_filesystem_resolver -> vllm.plugins.lora_resolvers.filesystem_resolver:register_filesystem_resolver
INFO 05-30 08:21:53 [__init__.py:36] All plugins in this group will be loaded. Set `VLL

In [ ]:
gc.collect()
torch.cuda.empty_cache()
quantize_and_evaluate("q4z")
gc.collect()
torch.cuda.empty_cache()

In [ ]:
gc.collect()
torch.cuda.empty_cache()
quantize_and_evaluate("q40")
gc.collect()
torch.cuda.empty_cache()

# Evaluation on Benchmarks

save quantized model to evaluate on benchmark with vLLM

In [ ]:
model = quantize_and_evaluate("q41")

In [ ]:
quantized_model_path = "qwen_7b_chat-weight_only_int4_quantization"
model.save_pretrained(quantized_model_path)
tokenizer.save_pretrained(quantized_model_path)

Evalute with the help of `lm-eval`

In [ ]:
!lm-eval --tasks ceval-valid --model vllm --model_args pretrained=./qwen_7b_chat-weight_only_int4_quantization,max_model_len=8192 --batch_size auto --trust_remote_code true